<a href="https://colab.research.google.com/github/jarekwan/PROJEKT_SCANNER/blob/main/10KONFIGURACJA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os
os.makedirs('/content/drive/MyDrive/projekt_test', exist_ok=True)

print("folder ready")

In [ ]:
%%writefile /content/drive/MyDrive/projekt_test/modul_konfiguracja.py

from __future__ import annotations

import json

from dataclasses import dataclass, field, fields
from enum import StrEnum, verify, UNIQUE
from pathlib import Path
from typing import Any, Final

from modul_filtry import (
    BaseFilter,
    FiltrSMA,
    FiltrMomentum,
    AdvancedMomentumFilter,
    FiltrZmiennosci,
)

from modul_skaner import (
    JsonScannerRepository,
    ScannerRepository,
)


FOLDER_DANYCH: Final[Path] = Path(
    "/content/drive/MyDrive/projekt_test"
)

DOMYSLNY_TYP_REPOZYTORIUM: Final[str] = "json"


@verify(UNIQUE)
class TypFiltraConfig(StrEnum):
    SMA = "sma"
    MOMENTUM = "momentum"
    ADVANCED_MOMENTUM = "advanced_momentum"
    ZMIENNOSC = "zmiennosc"


@verify(UNIQUE)
class TypRepozytorium(StrEnum):
    JSON = "json"


@dataclass(
    frozen=True,
    slots=True,
    kw_only=True,
)
class KonfiguracjaFiltra:

    typ: TypFiltraConfig = field(
        metadata={
            "opis": "typ filtra"
        }
    )

    aktywny: bool = field(
        default=True,
        metadata={
            "opis": "czy filtr jest aktywny"
        }
    )

    parametry: dict[str, Any] = field(
        default_factory=dict,
        metadata={
            "opis": "parametry filtra"
        }
    )


@dataclass(
    frozen=True,
    slots=True,
    kw_only=True,
)
class KonfiguracjaRepozytorium:

    typ: TypRepozytorium = field(
        default=TypRepozytorium.JSON,
        metadata={
            "opis": "typ repozytorium"
        }
    )

    folder: Path = field(
        default=FOLDER_DANYCH,
        metadata={
            "opis": "folder danych"
        }
    )


@dataclass(
    frozen=True,
    slots=True,
    kw_only=True,
)
class Konfiguracja:

    tickery: list[str] = field(
        default_factory=list,
        metadata={
            "opis": "lista analizowanych spolek"
        }
    )

    filtry: list[KonfiguracjaFiltra] = field(
        default_factory=list,
        metadata={
            "opis": "konfiguracja filtrow"
        }
    )

    repozytorium: KonfiguracjaRepozytorium = field(
        default_factory=KonfiguracjaRepozytorium,
        metadata={
            "opis": "konfiguracja repozytorium"
        }
    )

    def __post_init__(self) -> None:

        if not self.tickery:
            raise ValueError(
                "lista tickerow nie moze byc pusta"
            )

        if not self.filtry:
            raise ValueError(
                "lista filtrow nie moze byc pusta"
            )

        poprawione_tickery: list[str] = [
            ticker.strip().upper()
            for ticker in self.tickery
            if ticker.strip()
        ]

        if not poprawione_tickery:
            raise ValueError(
                "brak poprawnych tickerow"
            )

        object.__setattr__(
            self,
            "tickery",
            poprawione_tickery,
        )


DOMYSLNA_KONFIGURACJA: Final[
    dict[str, Any]
] = {

    "tickery": [
        "AAPL",
    ],

    "repozytorium": {
        "typ": "json",
        "folder": (
            "/content/drive/MyDrive/projekt_test"
        ),
    },

    "filtry": [

        {
            "typ": "sma",
            "aktywny": True,
            "parametry": {
                "minimalna_relacja": 1.0,
            },
        },

        {
            "typ": "momentum",
            "aktywny": True,
            "parametry": {
                "minimalne_momentum": 0.0,
            },
        },

        {
            "typ": "advanced_momentum",
            "aktywny": True,
            "parametry": {
                "minimalne_momentum": 2.0,
                "minimalny_wolumen": 100000.0,
            },
        },

        {
            "typ": "zmiennosc",
            "aktywny": True,
            "parametry": {
                "maksymalna_zmiennosc": 5.0,
            },
        },
    ],
}


class ConfigLoader:

    @classmethod
    def from_dict(
        cls,
        dane: dict[str, Any],
    ) -> Konfiguracja:

        tickery_raw: Any = dane.get(
            "tickery"
        )

        if not isinstance(
            tickery_raw,
            list,
        ):
            raise ValueError(
                "tickery musza byc lista"
            )

        tickery: list[str] = [
            str(ticker).strip().upper()
            for ticker in tickery_raw
            if str(ticker).strip()
        ]

        repo_raw: Any = dane.get(
            "repozytorium",
            {},
        )

        if not isinstance(
            repo_raw,
            dict,
        ):
            raise ValueError(
                "repozytorium musi byc dict"
            )

        try:
            typ_repo: TypRepozytorium = (
                TypRepozytorium(
                    str(
                        repo_raw.get(
                            "typ",
                            DOMYSLNY_TYP_REPOZYTORIUM,
                        )
                    )
                )
            )

        except ValueError as e:
            raise ValueError(
                "nieznany typ repozytorium"
            ) from e

        folder: Path = Path(
            str(
                repo_raw.get(
                    "folder",
                    FOLDER_DANYCH,
                )
            )
        )

        konfiguracja_repo = (
            KonfiguracjaRepozytorium(
                typ=typ_repo,
                folder=folder,
            )
        )

        filtry_raw: Any = dane.get(
            "filtry"
        )

        if not isinstance(
            filtry_raw,
            list,
        ):
            raise ValueError(
                "filtry musza byc lista"
            )

        filtry: list[
            KonfiguracjaFiltra
        ] = []

        for rekord in filtry_raw:

            if not isinstance(
                rekord,
                dict,
            ):
                raise ValueError(
                    "konfiguracja filtra "
                    "musi byc dict"
                )

            try:
                typ_filtra: TypFiltraConfig = (
                    TypFiltraConfig(
                        str(
                            rekord["typ"]
                        )
                    )
                )

            except (
                KeyError,
                ValueError,
            ) as e:
                raise ValueError(
                    "niepoprawny typ filtra"
                ) from e

            aktywny_raw: Any = (
                rekord.get(
                    "aktywny",
                    True,
                )
            )

            if not isinstance(
                aktywny_raw,
                bool,
            ):
                raise ValueError(
                    "aktywny musi byc bool"
                )

            parametry_raw: Any = (
                rekord.get(
                    "parametry",
                    {},
                )
            )

            if not isinstance(
                parametry_raw,
                dict,
            ):
                raise ValueError(
                    "parametry filtra "
                    "musza byc dict"
                )

            filtry.append(
                KonfiguracjaFiltra(
                    typ=typ_filtra,
                    aktywny=aktywny_raw,
                    parametry=dict(
                        parametry_raw
                    ),
                )
            )

        return Konfiguracja(
            tickery=tickery,
            filtry=filtry,
            repozytorium=(
                konfiguracja_repo
            ),
        )

    @classmethod
    def default(
        cls,
    ) -> Konfiguracja:

        return cls.from_dict(
            DOMYSLNA_KONFIGURACJA
        )

    @classmethod
    def from_json(
        cls,
        plik: Path,
    ) -> Konfiguracja:

        if not plik.exists():
            raise FileNotFoundError(
                f"brak pliku konfiguracji: "
                f"{plik}"
            )

        with open(
            plik,
            "r",
            encoding="utf-8",
        ) as f:

            dane: Any = json.load(
                f
            )

        if not isinstance(
            dane,
            dict,
        ):
            raise ValueError(
                "konfiguracja JSON "
                "musi byc obiektem"
            )

        return cls.from_dict(
            dane
        )


class FilterFactory:

    @classmethod
    def create(
        cls,
        config: KonfiguracjaFiltra,
    ) -> BaseFilter:

        if (
            config.typ
            == TypFiltraConfig.SMA
        ):

            return FiltrSMA(
                minimalna_relacja=float(
                    config.parametry.get(
                        "minimalna_relacja",
                        1.0,
                    )
                )
            )

        if (
            config.typ
            == TypFiltraConfig.MOMENTUM
        ):

            return FiltrMomentum(
                minimalne_momentum=float(
                    config.parametry.get(
                        "minimalne_momentum",
                        0.0,
                    )
                )
            )

        if (
            config.typ
            == TypFiltraConfig
            .ADVANCED_MOMENTUM
        ):

            return AdvancedMomentumFilter(

                minimalne_momentum=float(
                    config.parametry.get(
                        "minimalne_momentum",
                        2.0,
                    )
                ),

                minimalny_wolumen=float(
                    config.parametry.get(
                        "minimalny_wolumen",
                        100000.0,
                    )
                ),
            )

        if (
            config.typ
            == TypFiltraConfig.ZMIENNOSC
        ):

            return FiltrZmiennosci(
                maksymalna_zmiennosc=float(
                    config.parametry.get(
                        "maksymalna_zmiennosc",
                        5.0,
                    )
                )
            )

        raise ValueError(
            f"nieobslugiwany filtr: "
            f"{config.typ}"
        )

    @classmethod
    def create_active(
        cls,
        konfiguracja:
            list[KonfiguracjaFiltra],
    ) -> list[BaseFilter]:

        filtry: list[
            BaseFilter
        ] = []

        for config in konfiguracja:

            if not config.aktywny:
                continue

            filtr: BaseFilter = (
                cls.create(
                    config
                )
            )

            filtry.append(
                filtr
            )

        return filtry


class RepositoryFactory:

    @classmethod
    def create(
        cls,
        config:
            KonfiguracjaRepozytorium,
    ) -> ScannerRepository:

        if (
            config.typ
            == TypRepozytorium.JSON
        ):

            return JsonScannerRepository(
                folder=config.folder
            )

        raise ValueError(
            f"nieobslugiwane repozytorium: "
            f"{config.typ}"
        )


class FilterChainFactory:

    @classmethod
    def create(
        cls,
        konfiguracja:
            list[KonfiguracjaFiltra],
    ) -> BaseFilter:

        filtry: list[
            BaseFilter
        ] = (
            FilterFactory.create_active(
                konfiguracja
            )
        )

        if not filtry:
            raise ValueError(
                "brak aktywnych filtrow"
            )

        for aktualny, nastepny in zip(
            filtry,
            filtry[1:],
        ):

            aktualny.set_next(
                nastepny
            )

        return filtry[0]


@dataclass(
    slots=True,
    kw_only=True,
)
class ZestawKomponentow:

    konfiguracja: Konfiguracja

    repozytorium: ScannerRepository

    filtry: list[BaseFilter] = field(
        default_factory=list
    )

    pierwszy_filtr:
        BaseFilter | None = None


class ComponentFactory:

    @classmethod
    def create(
        cls,
        konfiguracja: Konfiguracja,
    ) -> ZestawKomponentow:

        repozytorium:
            ScannerRepository = (
                RepositoryFactory.create(
                    konfiguracja.repozytorium
                )
            )

        filtry: list[
            BaseFilter
        ] = (
            FilterFactory.create_active(
                konfiguracja.filtry
            )
        )

        if not filtry:
            raise ValueError(
                "brak aktywnych filtrow"
            )

        for aktualny, nastepny in zip(
            filtry,
            filtry[1:],
        ):

            aktualny.set_next(
                nastepny
            )

        return ZestawKomponentow(
            konfiguracja=konfiguracja,
            repozytorium=repozytorium,
            filtry=filtry,
            pierwszy_filtr=filtry[0],
        )


def pokaz_metadata(
    klasa: type[Any],
) -> None:

    print(
        "\nMETADATA:"
    )

    for f in fields(
        klasa
    ):

        print(
            f.name,
            "->",
            f.metadata.get(
                "opis",
                "brak opisu",
            ),
        )


def pokaz_konfiguracje(
    konfiguracja: Konfiguracja,
) -> None:

    print(
        "\nTICKERY:"
    )

    for ticker in konfiguracja.tickery:

        print(
            "-",
            ticker,
        )

    print(
        "\nFILTRY:"
    )

    for filtr in konfiguracja.filtry:

        print(
            filtr.typ.value,
            "| aktywny:",
            filtr.aktywny,
            "| parametry:",
            filtr.parametry,
        )

    print(
        "\nREPOZYTORIUM:"
    )

    print(
        konfiguracja
        .repozytorium
        .typ
        .value,
    )

    print(
        konfiguracja
        .repozytorium
        .folder,
    )


def run() -> None:

    print(
        "WCZYTYWANIE "
        "DOMYSLNEJ KONFIGURACJI"
    )

    konfiguracja: Konfiguracja = (
        ConfigLoader.default()
    )

    pokaz_konfiguracje(
        konfiguracja
    )

    komponenty: ZestawKomponentow = (
        ComponentFactory.create(
            konfiguracja
        )
    )

    print(
        "\nUTWORZONE FILTRY:"
    )

    for filtr in komponenty.filtry:

        print(
            "-",
            type(filtr).__name__,
        )

    print(
        "\nREPOZYTORIUM:"
    )

    print(
        type(
            komponenty.repozytorium
        ).__name__
    )

    print(
        "\nPIERWSZY FILTR LANCUCHA:"
    )

    if (
        komponenty.pierwszy_filtr
        is not None
    ):

        print(
            type(
                komponenty
                .pierwszy_filtr
            ).__name__
        )

    pokaz_metadata(
        Konfiguracja
    )

    pokaz_metadata(
        KonfiguracjaFiltra
    )

    print(
        "\nMODUL KONFIGURACJI "
        "DZIALA POPRAWNIE"
    )